# 🚀 ZONARETH-LM: Sleep-Proof GPU Training Studio (Kaggle & Colab)
### Train overnight safely, auto-save step progress, and download lightweight models

---

### 💤 HOW TO TRAIN WHILE SLEEPING (Zero Data Loss Guarantee)
If you go to sleep while running an interactive Kaggle tab, your browser may sleep and Kaggle can disconnect after 60 minutes.

**To train completely safely overnight:**
1. In the right panel under **Notebook options**:
   - Set **Accelerator** to **`GPU T4 x2`** or **`GPU P100`**.
   - Ensure **Internet** is toggled to **`On`**.
2. At the top right, click **`Save Version`** -> Choose **`Save & Run All (Commit)`**.
3. **You can now shut down your PC, close your browser, and go to sleep!**
4. Kaggle runs the notebook in a dedicated background container for up to 12 hours.
5. When you wake up, click your notebook and all generated files (`zonareth_125m_model.zip`, etc.) are permanently saved and ready in the **Output** section!

---

### 📦 THE 6GB ZIP SOLUTION (Why downloads were huge & how it's fixed)
- **The Problem:** Training saves full AdamW optimizer states (momentum + variance for every single parameter in FP32) across multiple steps, which ballooned the archive to 6 GB+ and risked running out of Kaggle's 20 GB disk limit.
- **The Solution:**
  1. **Rolling Step Checkpointing (`--max-to-keep 2`):** Progress is saved every 250 steps, keeping only the 2 latest step checkpoints + `best.pt` + `latest.pt`. Your progress is never lost, and disk never overflows!
  2. **Lightweight Release Archive (`zonareth_125m_model.zip`):** Pure FP16 model weights + tokenizer + config (~250 MB for 125M, ~700 MB for 350M). Downloads in seconds and is ready for `app.py`!
  3. **Full Resume Checkpoint (`zonareth_125m_resume_checkpoint.zip`):** Kept as a separate optional file with optimizer states for continuing training.

In [ ]:
# Step 1: Verify GPU Accelerator
import torch
print(f"PyTorch Version : {torch.__version__}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected  : {device_name} ({vram_gb:.2f} GB VRAM)")
    print(f"   CUDA Version  : {torch.version.cuda}")
else:
    print("⚠️ No GPU detected!")
    print("   On Kaggle: Right panel -> Notebook options -> Accelerator -> Select 'GPU T4 x2' or 'GPU P100'.")
    print("   On Colab: Runtime -> Change runtime type -> Select 'T4 GPU'.")

In [ ]:
# Step 2: Clone Repository & Install Dependencies
import os, sys
from pathlib import Path

if not Path("Veyra").exists() and not Path("Zonareth").exists() and not Path("configs").exists():
    !git clone https://github.com/shreyashmane-dev/Veyra.git
    %cd Veyra
elif Path("Veyra").exists() and not Path("configs").exists():
    %cd Veyra
elif Path("Zonareth").exists() and not Path("configs").exists():
    %cd Zonareth

!pip install -e . -q
!pip install datasets tqdm pyyaml huggingface_hub -q
print("\n✅ ZONARETH codebase & dependencies successfully installed!")

### 📂 Step 3: Dataset Ingestion & Preparation
Datasets are ingested from either:
- **Automatic Download (Zero Setup):** Downloads curated TinyStories or FineWeb-Edu.
- **Attached Kaggle Datasets:** Any dataset added via `+ Add Input` is automatically detected in `/kaggle/input/` and staged.

In [ ]:
# Step 3: Stage or Download Training Data
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# Check for attached Kaggle datasets
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    attached = [p for p in kaggle_input.iterdir() if p.is_dir()]
    if attached:
        print(f"✅ Detected {len(attached)} attached Kaggle dataset(s): {[p.name for p in attached]}")

# Ensure dataset is present
raw_files = list(raw_dir.glob("*.txt")) + list(raw_dir.glob("*.jsonl"))
if not raw_files:
    print("Downloading TinyStories pretraining corpus...")
    !python scripts/download_pretraining_data.py --dataset tinystories --max-mb 50
    raw_files = list(raw_dir.glob("*.txt")) + list(raw_dir.glob("*.jsonl"))

print(f"✅ Ready with {len(raw_files)} raw training file(s):")
for f in raw_files:
    print(f"   - {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)")

### ⚡ Step 4: Launch Training Loop with Step Saving & Rolling Checkpoints
The training runner will:
1. Train the Byte-Pair Encoding (`ZonarethTokenizer`).
2. Shard data into binary tokens for maximum GPU throughput.
3. **Save progress every 250 steps** (`--save-interval 250`).
4. Keep the **2 most recent rolling step checkpoints** (`--max-to-keep 2`) + `best.pt` + `latest.pt` so Kaggle disk is 100% protected from running out of space.
5. Automatically export the lightweight FP16 inference model (`model.pt`) upon completion.

In [ ]:
# Step 4: Run Training Pipeline
# For 125M model (recommended):
!python scripts/run_kaggle_pipeline.py --model 125m --steps 2500 --batch-size 8 --save-interval 250 --max-to-keep 2

# Note: To train the 350M model instead, run:
# !python scripts/run_kaggle_pipeline.py --model 350m --steps 2500 --batch-size 4 --save-interval 250 --max-to-keep 2

### 💬 Step 5: Test the Trained Model Directly in the Notebook
Load the clean trained weights and generate sample text immediately.

In [ ]:
# Step 5: Test Inference
from pathlib import Path
try:
    from zonareth.inference.engine import ZonarethInferenceEngine
except ImportError:
    from veyra.inference.engine import VeyraInferenceEngine as ZonarethInferenceEngine

ckpt_candidates = [
    Path("/kaggle/working/zonareth_125m_release/model.pt"),
    Path("/kaggle/working/zonareth_125m/model.pt"),
    Path("/kaggle/working/zonareth_125m/latest.pt"),
    Path("checkpoints/zonareth_125m/model.pt"),
    Path("checkpoints/zonareth_125m/latest.pt"),
]
ckpt_file = next((p for p in ckpt_candidates if p.exists()), None)
tok_path = Path("data/tokenizer")

if ckpt_file and tok_path.exists():
    print(f"✅ Loading trained model: {ckpt_file}")
    engine = ZonarethInferenceEngine.from_checkpoint(str(ckpt_file), str(tok_path))
    prompts = [
        "Once upon a time in a digital kingdom",
        "The secret algorithm began to learn",
        "In the deep forest of memories",
    ]
    print("\n" + "=" * 65)
    print("   ZONARETH-LM GENERATION TEST")
    print("=" * 65)
    for p in prompts:
        print(f"\nPrompt : {p}")
        res = engine.generate(p, max_new_tokens=50, temperature=0.8, top_k=40)
        print(f"Output : {res}")
else:
    print("Checkpoint not found. Run Step 4 first!")

### 📥 Step 6: Download the Model ZIP (Auto-Download & Manual Links)
This cell automatically triggers your browser download and creates direct links for the lightweight (~250 MB) package.

In [ ]:
# Step 6: Automatic Download & Verification
import os
from pathlib import Path
from IPython.display import display, HTML, Javascript, FileLink

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
model_zips = list(working_dir.glob("*_model.zip"))
all_zips = list(working_dir.glob("*.zip"))

target_zip = model_zips[0] if model_zips else (all_zips[0] if all_zips else None)

if target_zip:
    size_mb = target_zip.stat().st_size / (1024 * 1024)
    print("=" * 65)
    print(f"✅ READY FOR DOWNLOAD: {target_zip.name} ({size_mb:.2f} MB)")
    print("=" * 65)
    
    # List all available archives
    for z in all_zips:
        z_mb = z.stat().st_size / (1024 * 1024)
        is_rec = "(RECOMMENDED - Small & Fast)" if "_model.zip" in z.name else "(Full Resume Checkpoint)"
        print(f"   📦 {z.name} -> {z_mb:.2f} MB {is_rec}")

    # 1. Try Kaggle / Colab direct FileLink
    try:
        display(FileLink(str(target_zip.relative_to(Path.cwd())) if target_zip.is_relative_to(Path.cwd()) else str(target_zip.name)))
    except Exception:
        pass

    # 2. Trigger automatic browser download via JavaScript
    js_code = f"""
    var link = document.createElement('a');
    link.href = '/files/{target_zip.name}';
    link.download = '{target_zip.name}';
    document.body.appendChild(link);
    link.click();
    document.body.removeChild(link);
    console.log('Auto-download initiated for {target_zip.name}');
    """
    display(Javascript(js_code))
    
    # 3. HTML Download Button fallback
    html_button = f"""
    <div style='margin-top: 15px;'>
        <a href='/files/{target_zip.name}' download='{target_zip.name}'
           style='background-color: #2563eb; color: white; padding: 10px 20px; font-weight: bold; border-radius: 8px; text-decoration: none; display: inline-block;'>
           ⬇️ Click Here to Download {target_zip.name} ({size_mb:.1f} MB)
        </a>
    </div>
    <p style='color: gray; font-size: 13px; margin-top: 8px;'>
       Note: If you ran this via 'Save Version (Commit & Run)', you can also download anytime from the <b>Output</b> tab on the right sidebar!
    </p>
    """
    display(HTML(html_button))
else:
    print("No .zip archives found yet. Please run Step 4 first!")

### ☁️ Step 7: (Optional) Auto-Push Model to Hugging Face Hub
Want 100% cloud backup so you don't even need to download anything to your computer? Enter your Hugging Face write token below and push the entire model repository with one click.

In [ ]:
# Step 7: Optional Hugging Face Hub Upload
# Enter your HF Token and target repo name:
HF_TOKEN = ""  # e.g., 'hf_xxxxxxxxxxxxxxxxxxxxxxxx'
HF_REPO = ""   # e.g., 'your-username/zonareth-125m'

if HF_TOKEN and HF_REPO:
    from huggingface_hub import HfApi
    release_folder = next(Path("/kaggle/working").glob("*_release"), Path("zonareth_125m_release"))
    if release_folder.exists():
        print(f"Uploading {release_folder} to https://huggingface.co/{HF_REPO}...")
        api = HfApi(token=HF_TOKEN)
        api.create_repo(repo_id=HF_REPO, exist_ok=True, private=True)
        api.upload_folder(folder_path=str(release_folder), repo_id=HF_REPO)
        print(f"✅ Successfully uploaded to: https://huggingface.co/{HF_REPO}")
    else:
        print("Release folder not found. Run Step 4 first.")
else:
    print("HF_TOKEN and HF_REPO not set. Skipping cloud push (Local zip ready in Step 6).")